# Training Pipeline

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [6]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')

In [7]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [8]:
df.drop(columns=['id', 'Unnamed: 32'], inplace=True)

## Train-Test Split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:,0], test_size=0.2)

## Scaling

In [10]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)

## Encoding

In [11]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.fit_transform(y_test)

## Numpy Arrays to Torch

In [12]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

## Defining the Model

In [ ]:
class MySimpleNN:
    def __init__(self, X):
        self.weights = torch.rand(X.shape[1], 1, dtype=torch.float64, requires_grad=True)
        self.bias = torch.zeros(1, dtype=torch.float64, requires_grad=True)

    def forward(self, X):
        z = torch.matmul(X, self.weights) + self.bias 
        y_pred = torch.sigmoid(z)
        return y_pred
    
    def loss_function(self,y_pred, y):
        epsilon = 1e-7
        y_pred = torch.clamp(y_pred, epsilon, 1-epsilon)
        loss = -(y_train_tensor*torch.log(y_pred) + (1-y_train_tensor)*torch.log(1-y_pred)).mean()
        return loss

In [42]:
learning_rate= 0.3
epochs = 1000

## Training Pipeline

In [43]:
#create model
model = MySimpleNN(X_train_tensor)

#defineloop
for epoch in range(epochs):
    #forwardpass
    y_pred = model.forward(X_train_tensor)

    #losscalculate
    loss = model.loss_function(y_pred, y_train_tensor)
    
    #backwardpass
    loss.backward()

    #parametersupdate
    with torch.no_grad():
        model.weights -= learning_rate*model.weights.grad
        model.bias -= learning_rate * model.bias.grad
    
    #zerogradients
    model.weights.grad.zero_()
    model.bias.grad.zero_()

    print (f'Epoch: {epoch + 1}, Loss: {loss.item()}')


Epoch: 1, Loss: 3.8962086561679765
Epoch: 2, Loss: 3.551091660237547
Epoch: 3, Loss: 3.164163023876579
Epoch: 4, Loss: 2.7169308278518454
Epoch: 5, Loss: 2.2393171776519023
Epoch: 6, Loss: 1.7442194869242627
Epoch: 7, Loss: 1.3006598714133193
Epoch: 8, Loss: 0.9731722347505986
Epoch: 9, Loss: 0.8269830418116254
Epoch: 10, Loss: 0.7825935199295125
Epoch: 11, Loss: 0.761459539656636
Epoch: 12, Loss: 0.7456210272767243
Epoch: 13, Loss: 0.732918109861871
Epoch: 14, Loss: 0.7227117543317846
Epoch: 15, Loss: 0.7145210443719018
Epoch: 16, Loss: 0.7079433898112345
Epoch: 17, Loss: 0.702644458606477
Epoch: 18, Loss: 0.6983505935388176
Epoch: 19, Loss: 0.6948414808223181
Epoch: 20, Loss: 0.6919430725734962
Epoch: 21, Loss: 0.689520477057361
Epoch: 22, Loss: 0.687470785731412
Epoch: 23, Loss: 0.6857162372835205
Epoch: 24, Loss: 0.6841982108063751
Epoch: 25, Loss: 0.6828723266009534
Epoch: 26, Loss: 0.6817046710866145
Epoch: 27, Loss: 0.6806689979403839
Epoch: 28, Loss: 0.6797447006143015
Epoch: 2

In [44]:
model.weights

tensor([[-0.0691],
        [ 0.0109],
        [ 0.2732],
        [-0.2328],
        [ 0.0024],
        [-0.1492],
        [ 0.0730],
        [ 0.0073],
        [ 0.0110],
        [ 0.0491],
        [-0.0577],
        [-0.0007],
        [ 0.0292],
        [ 0.0404],
        [ 0.0068],
        [-0.0083],
        [ 0.0317],
        [-0.0203],
        [ 0.0094],
        [-0.0072],
        [ 0.0578],
        [-0.0130],
        [ 0.0033],
        [-0.0103],
        [ 0.0025],
        [ 0.1087],
        [-0.1038],
        [ 0.0269],
        [-0.0194],
        [-0.0201]], dtype=torch.float64, requires_grad=True)

In [45]:
model.bias

tensor([-0.4701], dtype=torch.float64, requires_grad=True)

In [46]:
with torch.no_grad():
    y_pred = model.forward(X_test_tensor)
    y_pred = (y_pred > 0.5).float()

print(y_pred)

tensor([[0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
      

In [47]:
accuracy = (y_pred == y_test_tensor).float().mean()
print(f'Accuracy: {accuracy.item()}')

Accuracy: 0.6754385828971863
